[![Open In Colab](https://google.com)](https://colab.research.google.com/github/hamagami/IS2026/blob/main/98_GMM_EM_Interactive_JA_EN.ipynb)

[![Open In Colab](https://google.com)](https://google.comhamagami/IS2026/blob/main/98_GMM_EM_Interactive_JA_EN.ipynb)

# GMMとEMアルゴリズム：インタラクティブ実験 / GMM and EM: an interactive demonstration

**操作**：Colabにアップロードし，「ランタイム → すべてのセルを実行」を選びます．CPUで動き，データのダウンロードや認証は不要です．

**How to run:** Upload to Colab and select Runtime → Run all. Runs on a CPU without dataset downloads or authentication.

「次のステップ」でEとMを交互に進めます．点の位置は動かず，所属確率と分布のパラメータが変わります．

Use Next step to alternate E and M. Data points stay fixed while responsibilities and distribution parameters change.

タイトル，説明，コメント，図と操作は日本語・英語併記です．初回のパッケージ・日本語フォントの準備には通信を使います．

Titles, explanations, comments, figures, and controls are bilingual. Initial package/font installation uses internet access.

## 1．モデルと更新式 / Model and update equations

データの分布をK個のガウス分布の混合で表します．πは混合比率，μは平均，Σは共分散です．

Represent the data with K Gaussian components. π denotes mixture weights, μ means, and Σ covariances.

$$p(x_i)=\sum_{k=1}^K\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k),\qquad \sum_k\pi_k=1.$$

**Eステップ：** 現在のパラメータを固定し，所属確率（負担率）rを計算します．真の種類の確率ではなく，現在のモデルに基づく成分への所属確率です．

**E-step:** Hold the current parameters fixed and compute responsibilities r: posterior component probabilities under the current model, not probabilities of ground-truth semantic classes.

$$r_{ik}=\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal N(x_i\mid\mu_j,\Sigma_j)}.$$

**Mステップ：** rを重みとして平均，共分散，混合比率を更新します．点を一つの成分に決める処理ではありません．

**M-step:** Use r as weights to update means, covariances, and mixture weights. Points are not assigned exclusively to one component.

$$N_k=\sum_i r_{ik},\quad \pi_k^{new}=N_k/N,\quad \mu_k^{new}=\frac{\sum_i r_{ik}x_i}{N_k},$$
$$\Sigma_k^{new}=\frac{\sum_i r_{ik}(x_i-\mu_k^{new})(x_i-\mu_k^{new})^T}{N_k}.$$

観測データの対数尤度を確認します．Eでは分布を変えないため，この値は変わりません．Mの後に変化します．

Track the observed-data log likelihood. It stays unchanged in E because distribution parameters do not change, and changes after M.

$$\ell=\sum_i\log\left[\sum_k\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)\right].$$

数値安定化のため，共分散の固有値に下限10⁻⁴を設けます．Mでは固有値を下限で切り上げます．これは共分散の下限制約を加えた更新です．

For numerical stability, covariance eigenvalues are constrained to be at least 10⁻⁴. The M-step clips eigenvalues to that floor: a constrained covariance update.

## 2．準備 / Setup

In [ ]:
import sys, subprocess, importlib.util
# 不足するパッケージのみ追加します． / Install only missing packages.
for module, package in [('numpy','numpy'), ('scipy','scipy'), ('matplotlib','matplotlib'),
                        ('ipywidgets','ipywidgets'), ('japanize_matplotlib','japanize-matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.special import logsumexp
import ipywidgets as widgets
from IPython.display import display, clear_output
import japanize_matplotlib
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
plt.rcParams.update({'figure.dpi':110, 'font.size':10, 'axes.unicode_minus':False})
COLORS=np.array([[.97,.48,.08],[.04,.65,.65],[.55,.22,.8],[.15,.45,.85],[.8,.2,.4]])
N=240
EIGEN_FLOOR=1e-4

## 3．平面のデータ / Two-dimensional data

In [ ]:
def make_data(kind='ellipses'):
    # データ用乱数を固定し，初期値だけの違いを比べます． / Fix data randomness to isolate initialization effects.
    rng=np.random.default_rng(12)
    if kind=='moons':
        t=np.linspace(0,np.pi,N//2)
        X=np.vstack([np.column_stack([np.cos(t),np.sin(t)]),
                     np.column_stack([1-np.cos(t),.5-np.sin(t)])])
        X+=rng.normal(0,.06,X.shape)
    else:
        means=np.array([[-2,1.3],[1.9,1.1],[0,-1.8]])
        covs=np.array([[[1.1,.65],[.65,.6]],[[.55,-.4],[-.4,.95]],[[.85,.1],[.1,.22]]])
        if kind=='overlap':means*=.45
        X=np.vstack([rng.multivariate_normal(m,c,n) for m,c,n in zip(means,covs,[100,80,60])])
    return X[rng.permutation(len(X))]
X=make_data()
plt.figure(figsize=(6,4));plt.scatter(*X.T,s=20,color='slategray')
plt.xlabel('特徴量1 / Feature 1');plt.ylabel('特徴量2 / Feature 2')
plt.title('観測データ：ラベルは使用しない / Observations: no labels used');plt.show()

## 4．EMの実装 / EM implementation

In [ ]:
class EMGMM:
    def reset(self, kind='ellipses', K=3, seed=0):
        self.X=make_data(kind);self.K=K
        rng=np.random.default_rng(seed)
        self.means=self.X[rng.choice(len(self.X),K,replace=False)].copy()
        self.covs=np.repeat(np.cov(self.X,rowvar=False)[None,:,:],K,axis=0)
        self.weights=np.full(K,1/K)
        self.r=None;self.next_phase='E';self.iteration=0;self.movement=None
        self.stage='初期化：次はE / Initialized: E is next'
        self.trace=[self.log_likelihood()]

    def log_joint(self,X):
        # 密度を対数空間で計算し，アンダーフローを避けます． / Work in log space to avoid density underflow.
        X=np.asarray(X);result=[]
        for k in range(self.K):
            diff=X-self.means[k];L=np.linalg.cholesky(self.covs[k])
            v=np.linalg.solve(L,diff.T)
            logpdf=-.5*(X.shape[1]*np.log(2*np.pi)+2*np.log(np.diag(L)).sum()+(v*v).sum(axis=0))
            result.append(np.log(self.weights[k])+logpdf)
        return np.column_stack(result)

    def posterior(self,X):
        logp=self.log_joint(X)
        return np.exp(logp-logsumexp(logp,axis=1,keepdims=True))

    def log_likelihood(self):
        return float(logsumexp(self.log_joint(self.X),axis=1).sum())

    def step(self):
        if self.next_phase=='E':
            # E：パラメータは固定し，所属確率だけ更新します． / E: update responsibilities with parameters fixed.
            self.r=self.posterior(self.X)
            self.next_phase='M';self.movement=None
            self.stage='E：所属確率を更新 / E: updated responsibilities'
        else:
            # M：直前のEの確率を固定して，分布を更新します． / M: update parameters with the latest E responsibilities fixed.
            Nk=self.r.sum(axis=0)
            if np.any(Nk<1e-12):
                raise ValueError('成分の重みが極小です．初期化してください． / Tiny component: reset the demo.')
            old=self.means.copy();self.weights=Nk/len(self.X)
            self.means=self.r.T@self.X/Nk[:,None]
            for k in range(self.K):
                diff=self.X-self.means[k]
                cov=(diff.T*self.r[:,k])@diff/Nk[k]
                values,vectors=np.linalg.eigh((cov+cov.T)/2)
                self.covs[k]=(vectors*np.maximum(values,EIGEN_FLOOR))@vectors.T
            self.iteration+=1;self.next_phase='E';self.movement=(old,self.means.copy())
            self.trace.append(self.log_likelihood())
            self.stage='M：平均・共分散・混合比率を更新 / M: updated means, covariances, weights'
        return self.stage

    def run(self, iterations=10):
        # M待ちならそのMから再開し，指定回数のM更新を行います． / If M is pending, resume there and perform the requested number of M updates.
        target=self.iteration+iterations
        while self.iteration<target:self.step()

    def sample(self,n=240,seed=2026):
        # 成分を混合比率で選んでから，そのガウス分布から生成します． / Choose a component by its weight, then sample its Gaussian.
        rng=np.random.default_rng(seed);component=rng.choice(self.K,size=n,p=self.weights)
        out=np.empty((n,2))
        for k in range(self.K):
            mask=component==k
            out[mask]=rng.multivariate_normal(self.means[k],self.covs[k],size=mask.sum())
        return out,component

demo=EMGMM();demo.reset()

## 5．一段ずつ観察 / Explore step by step

- **初期化 → 次のステップ → 次のステップ**でE→Mを進めます．成分数や初期値を変えたら「初期化」を押します．
  **Reset → Next step → Next step** advances E then M. Press Reset after changing component count or initialization.
- **Eでは点の色，Mでは楕円・中心・混合比率**に注目します．色は所属確率の混色です．Mの表示では，直前のEの色と棒グラフを保ちます．新しいパラメータによる所属確率は次のEで表示します．
  **E changes colors; M changes ellipses, means, and weights.** Colors blend responsibilities. During M, colors and bars retain the previous E responsibilities; the next E displays probabilities for the new parameters.
- **注目点**を動かすと，黒い輪で囲んだ点の所属確率が棒グラフに出ます．楕円は各成分の2標準偏差の等密度輪郭であり，所属の境界ではありません．
  Change **Point** to inspect the circled observation's responsibilities. Ellipses are component contours at two standard deviations, not membership boundaries.
- **10回反復**で進め，対数尤度が増える様子を確認します．初期値や成分数によって結果は変わり，大域最適解は保証されません．
  Use **Run 10** and inspect log-likelihood improvement. Initialization and component count affect results; global optimality is not guaranteed.

In [ ]:
def draw_ellipses(ax):
    for k in range(demo.K):
        vals,vecs=np.linalg.eigh(demo.covs[k]);order=vals.argsort()[::-1]
        vals=vals[order];vecs=vecs[:,order]
        angle=np.degrees(np.arctan2(vecs[1,0],vecs[0,0]))
        ax.add_patch(Ellipse(demo.means[k],4*np.sqrt(vals[0]),4*np.sqrt(vals[1]),
                             angle=angle,fill=False,lw=2,color=COLORS[k]))
        ax.scatter(*demo.means[k],marker='X',s=140,color=COLORS[k],edgecolor='black',zorder=4)

def draw(point=0):
    fig,axes=plt.subplots(2,2,figsize=(12,8),gridspec_kw={'width_ratios':[1.65,1]})
    ax=axes[0,0]
    color='slategray' if demo.r is None else demo.r@COLORS[:demo.K]
    ax.scatter(*demo.X.T,c=color,s=23,alpha=.85);draw_ellipses(ax)
    ax.scatter(*demo.X[point],s=160,facecolors='none',edgecolors='black',lw=2,zorder=5)
    if demo.movement is not None:
        for a,b in zip(*demo.movement):ax.annotate('',xy=b,xytext=a,arrowprops={'arrowstyle':'->','color':'black'})
    pad=1.;lo=demo.X.min(0)-pad;hi=demo.X.max(0)+pad
    ax.set_xlim(lo[0],hi[0]);ax.set_ylim(lo[1],hi[1]);ax.set_aspect('equal',adjustable='box')
    ax.set_xlabel('特徴量1 / Feature 1');ax.set_ylabel('特徴量2 / Feature 2')
    ax.set_title('点は固定・色は所属確率 / Fixed points; colors = responsibilities')
    ax=axes[0,1];labels=[f'C{k+1}' for k in range(demo.K)]
    if demo.r is not None:
        bars=ax.bar(labels,demo.r[point],color=COLORS[:demo.K])
        for bar,value in zip(bars,demo.r[point]):ax.text(bar.get_x()+bar.get_width()/2,value+.02,f'{value:.2f}',ha='center')
    else:ax.text(.5,.5,'Eを実行してください / Run E first',ha='center',transform=ax.transAxes)
    ax.set_ylim(0,1.12);ax.set_ylabel('所属確率 / Responsibility')
    ax.set_title(f'注目点 / Point {point}\n直前Eの確率 / Latest E-step probabilities')
    ax=axes[1,0];ax.plot(range(len(demo.trace)),demo.trace,'o-',color='#1478d8')
    ax.set_xlabel('M更新回数 / M updates');ax.set_ylabel('対数尤度 / Log likelihood')
    ax.set_title('Eは分布を固定・Mで更新 / E holds distributions; M updates them')
    ax=axes[1,1];ax.axis('off')
    rows=[]
    for k in range(demo.K):
        rows.append([f'C{k+1}',f'{demo.weights[k]:.3f}',
                     f'({demo.means[k,0]:.2f}, {demo.means[k,1]:.2f})',
                     f'{np.linalg.eigvalsh(demo.covs[k]).min():.3f}'])
    table=ax.table(cellText=rows,colLabels=['成分\nComponent','混合比率\nWeight π','平均\nMean μ','最小固有値\nMin eig(Σ)'],loc='center',cellLoc='center')
    table.auto_set_font_size(False);table.set_fontsize(9);table.scale(1,2.1)
    ax.set_title('現在のパラメータ / Current parameters')
    fig.suptitle(f'{demo.stage}\nM更新 / M updates: {demo.iteration}   次 / Next: {demo.next_phase}',fontsize=13)
    fig.tight_layout(rect=[0,0,1,.92]);plt.show()

style={'description_width':'initial'}
kind=widgets.Dropdown(options=[('細長いガウス / Ellipses','ellipses'),('重なるガウス / Overlap','overlap'),('三日月：制約の例 / Moons: limitation','moons')],description='データ / Data',style=style,layout=widgets.Layout(width='420px'))
K_control=widgets.IntSlider(value=3,min=2,max=5,description='成分数 / K',style=style)
seed=widgets.IntSlider(value=0,min=0,max=20,description='初期値 / Seed',style=style)
point=widgets.IntSlider(value=0,min=0,max=N-1,description='注目点 / Point',style=style,continuous_update=False)
reset_button=widgets.Button(description='初期化 / Reset',button_style='info',layout=widgets.Layout(width='180px'))
next_button=widgets.Button(description='次のステップ / Next',button_style='success',layout=widgets.Layout(width='210px'))
run_button=widgets.Button(description='10回反復 / Run 10',layout=widgets.Layout(width='200px'))
out=widgets.Output()
def refresh(_=None):
    with out:clear_output(wait=True);draw(point.value)
def reset_click(_=None):
    demo.reset(kind.value,K_control.value,seed.value);refresh()
def next_click(_=None):
    demo.step();refresh()
def run_click(_=None):
    demo.run(10);refresh()
reset_button.on_click(reset_click);next_button.on_click(next_click);run_button.on_click(run_click)
point.observe(refresh,names='value')
display(widgets.VBox([kind,widgets.HBox([K_control,seed]),point,
                      widgets.HBox([reset_button,next_button,run_button]),out]))
refresh()

## 6．学習した分布から生成 / Generate from the learned distribution

いくつかのEM更新を行ってから，このセルのボタンを押します．成分を混合比率で選び，その成分のガウス分布から新しい点を生成します．生成ラベルはサンプリング時に選んだ成分であり，観測データの正解ラベルではありません．

After several EM updates, press the button in this cell. Select a component by its mixture weight, then draw a new point from that Gaussian. Sample labels identify the sampled components, not ground-truth labels for observations.

In [ ]:
def show_samples(seed=2026):
    generated,components=demo.sample(seed=seed)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax in axes:draw_ellipses(ax);ax.set_xlabel('特徴量1 / Feature 1');ax.set_ylabel('特徴量2 / Feature 2')
    axes[0].scatter(*demo.X.T,s=20,color='slategray',alpha=.65)
    axes[0].set_title('観測データ / Observed data')
    axes[1].scatter(*generated.T,s=20,c=COLORS[components],alpha=.7)
    axes[1].set_title('GMMからの新規サンプル / New GMM samples')
    both=np.vstack([demo.X,generated]);lo=both.min(0)-.8;hi=both.max(0)+.8
    for ax in axes:ax.set_xlim(lo[0],hi[0]);ax.set_ylim(lo[1],hi[1]);ax.set_aspect('equal',adjustable='box')
    fig.tight_layout();plt.show()
sampling_ui=widgets.interactive(show_samples,{'manual':True,'manual_name':'生成 / Generate'},
    seed=widgets.IntSlider(value=2026,min=0,max=9999,description='生成乱数 / Seed',style=style))
display(sampling_ui)

## 7．ウィジェットなしで実行 / Run without widgets

表示されない場合，次のセルを手動で実行できます．このセルはそれまでの実験を初期化します．

If widgets are unavailable, run the following cell manually. It resets the current experiment.

In [ ]:
# 必要な場合のみ実行します．コメントを外してください． / Uncomment and run only if needed.
# demo.reset('ellipses', K=3, seed=0)
# demo.step(); draw(point=0)  # E：所属確率 / E: responsibilities
# demo.step(); draw(point=0)  # M：パラメータ / M: parameters
# demo.run(20); draw(point=0)  # 20回のM更新 / 20 M updates
# show_samples(seed=2026)  # 新規データの生成 / Generate new data

## 8．講義中の問い / Questions for discussion

1. 重なるデータで，境界付近の点は複数の成分にどの程度所属していますか．
   With overlapping data, how is responsibility divided for boundary points?
2. 成分数を2，3，5に変えると，分布の形と対数尤度はどう変わりますか．尤度だけで成分数を決められるでしょうか．
   How do shapes and likelihood change with 2, 3, and 5 components? Is training likelihood alone enough to choose K?
3. 初期値を変えて同じ回数更新すると，同じ結果になりますか．成分番号の入れ替わりと分布自体の違いを区別してください．
   Do different initializations reach the same fit? Distinguish label permutations from genuinely different distributions.
4. 三日月データはガウスの混合で密度を近似できますが，各成分が見た目のグループ一つに対応するとは限りません．
   Mixtures can approximate the density of moons, but individual components need not match the visible groups.

初期値やKの変更は初期化時に反映します．Eだけでは平均・共分散・混合比率は変わりません．モデルに基づく所属の曖昧さと，パラメータ自体の推定不確実性は異なります．このデモは後者を推定していません．

Changes to initialization or K take effect on Reset. E alone does not change means, covariances, or weights. Uncertain component membership differs from uncertainty in parameter estimates; the latter is not estimated here.

## 参考 / References

- [scikit-learn：混合ガウスモデル / Gaussian mixture models](https://scikit-learn.org/stable/modules/mixture.html)
- [SciPy：対数和指数 / logsumexp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.logsumexp.html)

ローカルCPUでコードと操作処理を検証しています．Colabのブラウザー画面そのものでは未検証です．

Code and control callbacks are validated on a local CPU; the Colab browser interface itself has not been tested.